In [ ]:
from pathlib import Path
import os, sys
for project_root in (Path.cwd(), *Path.cwd().parents):
    if (project_root / 'regarch').is_dir():
        os.chdir(project_root)
        sys.path.insert(0, str(project_root))
        break
else:
    raise RuntimeError('Run this notebook from inside the RegArch project.')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from regarch import FateMap, GRNInference, load_fate_map_topology, parse_edge_dict
from regarch.downstream_analysis import (
    cluster_regulatory_interactions, filter_regulators, get_high_expression_genes,
    identify_differentiation_regulators, identify_fate_bias_regulators,
    merge_regulator_scores, summarize_interaction_cluster_specificity,
)
from regarch.plotting import (
    plot_dynamic_regulator_count, plot_key_genes_differentiation,
    plot_key_genes_fate_bias, plot_regulatory_genes_along_fate_map,
    plot_regulatory_interactions_along_fate_map,
    plot_regulatory_interactions_by_cell_type,
    plot_regulatory_network_along_fate_map,
    plot_target_genes_along_fate_map,
)
from regarch.utils.data_io import (
    export_grn_edges, get_dynamic_networks,
)
from regarch.utils.metrics import (
    count_greater_than_threshold, match_expression_data, normalize_to_zero_mean,
)

# Set True to recompute GRNs; False uses the bundled reference outputs.
RECOMPUTE_GRNS = False

## Infer GRNs

## Validate the performance of RegArch on C.elegans dataset

In [ ]:
target_genes_name=pd.read_csv("examples/data/C.elegans/C.elegans_Full/additional_input_data/target_genes_name.csv").iloc[:,1].to_list()
regulatory_genes_name=pd.read_csv("examples/data/C.elegans/C.elegans_Full/additional_input_data/regulatory_genes_name.csv").iloc[:,1].to_list()

output_path='examples/results/C.elegans/C.elegans_Full/figures/'

tar_num=len(target_genes_name)

In [ ]:
base_path = 'examples/data/C.elegans/C.elegans_Full/additional_input_data'

file_names = [
    'ABprp_count_matrix.csv',
    'ABprap_count_matrix.csv',
    'ABara_count_matrix.csv',
    'ABaraap_count_matrix.csv',
    'ABplpa_count_matrix.csv',
    'ABalp_count_matrix.csv',
    'ABala_count_matrix.csv',
    'ABplapp_count_matrix.csv',
    'ABplp_count_matrix.csv',
    'ABarp_count_matrix.csv',
    'ABarpa_count_matrix.csv',
    'ABprpa_count_matrix.csv'
]
count_matrices = {}

for file in file_names:
    key = os.path.splitext(file)[0]
    count_matrices[key] = pd.read_csv(os.path.join(base_path, file), index_col=0)

#### High confidence dataset verification

In [ ]:
ABprp_norm_count_matrix=normalize_to_zero_mean(count_matrices['ABprp_count_matrix'])
ABprp_tar=ABprp_norm_count_matrix.iloc[range(tar_num),:]
ABprp_reg=ABprp_norm_count_matrix.iloc[tar_num:,:]

ABprap_norm_count_matrix=normalize_to_zero_mean(count_matrices['ABprap_count_matrix'])
ABprap_tar=ABprap_norm_count_matrix.iloc[range(tar_num),:]
ABprap_reg=ABprap_norm_count_matrix.iloc[tar_num:,:]

ABara_norm_count_matrix=normalize_to_zero_mean(count_matrices['ABara_count_matrix'])
ABara_tar=ABara_norm_count_matrix.iloc[range(tar_num),:]
ABara_reg=ABara_norm_count_matrix.iloc[tar_num:,:]

ABaraap_norm_count_matrix=normalize_to_zero_mean(count_matrices['ABaraap_count_matrix'])
ABaraap_tar=ABaraap_norm_count_matrix.iloc[range(tar_num),:]
ABaraap_reg=ABaraap_norm_count_matrix.iloc[tar_num:,:]

ABplpa_norm_count_matrix=normalize_to_zero_mean(count_matrices['ABplpa_count_matrix'])
ABplpa_tar=ABplpa_norm_count_matrix.iloc[range(tar_num),:]
ABplpa_reg=ABplpa_norm_count_matrix.iloc[tar_num:,:]

ABalp_norm_count_matrix=normalize_to_zero_mean(count_matrices['ABalp_count_matrix'])
ABalp_tar=ABalp_norm_count_matrix.iloc[range(tar_num),:]
ABalp_reg=ABalp_norm_count_matrix.iloc[tar_num:,:]

In [ ]:
fate_map_df = pd.read_csv('examples/data/C.elegans/C.elegans_Full/additional_input_data/fate_map_topology.csv')
edges_dict_Celegans = {}
    
for top_key, group in fate_map_df .groupby("Lineage"):
    sub_dict = {f"{row['Parent']}->{row['Child']}": row['Length'] for _, row in group.iterrows()}
    edges_dict_Celegans[top_key] = sub_dict

In [ ]:
edges_ABpr = parse_edge_dict(edges_dict_Celegans['ABpr'])
fate_map_ABpr = FateMap(edges_ABpr)
atac_file_path_ABpr = "examples/data/C.elegans/C.elegans_Full/input_data/ABpr/atac_data.csv"
expression_file_path_ABpr = "examples/data/C.elegans/C.elegans_Full/input_data/ABpr/expression_data.csv"

saved_dir_ABpr = 'examples/results/C.elegans/C.elegans_Full/ABpr'

if RECOMPUTE_GRNS:
    grn_inference_result = GRNInference(atac_file_path_ABpr, expression_file_path_ABpr, fate_map_ABpr, saved_dir_ABpr)
    grn_inference_result.infer_grn(20)

ABpr_grn_dict = get_dynamic_networks(saved_dir_ABpr, fate_map_ABpr, 0, regulatory_genes_name, target_genes_name)

## ABpl
edges_ABpl = parse_edge_dict(edges_dict_Celegans['ABpl'])
fate_map_ABpl = FateMap(edges_ABpl)
atac_file_path_ABpl = "examples/data/C.elegans/C.elegans_Full/input_data/ABpl/atac_data.csv"
expression_file_path_ABpl = "examples/data/C.elegans/C.elegans_Full/input_data/ABpl/expression_data.csv"

saved_dir_ABpl = 'examples/results/C.elegans/C.elegans_Full/ABpl'

if RECOMPUTE_GRNS:
    grn_inference_result = GRNInference(atac_file_path_ABpl, expression_file_path_ABpl, fate_map_ABpl, saved_dir_ABpl)
    grn_inference_result.infer_grn(20)

ABpl_grn_dict = get_dynamic_networks(saved_dir_ABpl, fate_map_ABpl, 0, regulatory_genes_name, target_genes_name)

## ABar
edges_ABar = parse_edge_dict(edges_dict_Celegans['ABar'])
fate_map_ABar = FateMap(edges_ABar)
atac_file_path_ABar = "examples/data/C.elegans/C.elegans_Full/input_data/ABar/atac_data.csv"
expression_file_path_ABar = "examples/data/C.elegans/C.elegans_Full/input_data/ABar/expression_data.csv"

saved_dir_ABar = 'examples/results/C.elegans/C.elegans_Full/ABar'

if RECOMPUTE_GRNS:
    grn_inference_result = GRNInference(atac_file_path_ABar, expression_file_path_ABar, fate_map_ABar, saved_dir_ABar)
    grn_inference_result.infer_grn(20)

ABar_grn_dict = get_dynamic_networks(saved_dir_ABar,fate_map_ABar,0,regulatory_genes_name, target_genes_name)

## ABal
edges_ABal = parse_edge_dict(edges_dict_Celegans['ABal'])
fate_map_ABal= FateMap(edges_ABal)
atac_file_path_ABal = "examples/data/C.elegans/C.elegans_Full/input_data/ABal/atac_data.csv"
expression_file_path_ABal = "examples/data/C.elegans/C.elegans_Full/input_data/ABal/expression_data.csv"

saved_dir_ABal = 'examples/results/C.elegans/C.elegans_Full/ABal'

if RECOMPUTE_GRNS:
    grn_inference_result = GRNInference(atac_file_path_ABal, expression_file_path_ABal, fate_map_ABal, saved_dir_ABal)
    grn_inference_result.infer_grn(20)

ABal_grn_dict = get_dynamic_networks(saved_dir_ABal, fate_map_ABal, 0, regulatory_genes_name, target_genes_name)



In [ ]:
## EMS
edges_EMS = parse_edge_dict(edges_dict_Celegans['EMS'])
fate_map_EMS = FateMap(edges_EMS)
atac_file_path_EMS = "examples/data/C.elegans/C.elegans_Full/input_data/EMS/atac_data.csv"
expression_file_path_EMS = "examples/data/C.elegans/C.elegans_Full/input_data/EMS/expression_data.csv"

saved_dir_EMS = 'examples/results/C.elegans/C.elegans_Full/EMS'

if RECOMPUTE_GRNS:
    grn_inference_result = GRNInference(atac_file_path_EMS, expression_file_path_EMS, fate_map_EMS, saved_dir_EMS)
    grn_inference_result.infer_grn(20)

EMS_grn_dict = get_dynamic_networks(saved_dir_EMS, fate_map_EMS, 0, regulatory_genes_name, target_genes_name)

## P2
edges_P2 = parse_edge_dict(edges_dict_Celegans['P2'])
fate_map_P2 = FateMap(edges_P2)
atac_file_path_P2 = "examples/data/C.elegans/C.elegans_Full/input_data/P2/atac_data.csv"
expression_file_path_P2 = "examples/data/C.elegans/C.elegans_Full/input_data/P2/expression_data.csv"

saved_dir_P2 = 'examples/results/C.elegans/C.elegans_Full/P2'

if RECOMPUTE_GRNS:
    grn_inference_result = GRNInference(atac_file_path_P2, expression_file_path_P2, fate_map_P2, saved_dir_P2)
    grn_inference_result.infer_grn(20)

P2_grn_dict = get_dynamic_networks(saved_dir_P2, fate_map_P2, 0, regulatory_genes_name, target_genes_name)

In [ ]:
grn_sources = {
    'ABprp': ABpr_grn_dict,
    'ABprap': ABpr_grn_dict,
    'ABara': ABar_grn_dict,
    'ABaraap': ABar_grn_dict,
    'ABplpa': ABpl_grn_dict,
    'ABalp': ABal_grn_dict,
    'ABala': ABal_grn_dict,
    'E': EMS_grn_dict,
    'ABal': ABal_grn_dict,
    'EMS': EMS_grn_dict,
    'MS': EMS_grn_dict,
    'ABarp': ABar_grn_dict,
    'ABar': ABar_grn_dict,
    'ABarpa': ABar_grn_dict,
    'ABaraa': ABar_grn_dict,
    'ABpr': ABpr_grn_dict,
    'ABprpa': ABpr_grn_dict,
    'ABpra': ABpr_grn_dict,
    'ABpl': ABpl_grn_dict,
    'ABplp': ABpl_grn_dict,
    'ABpla': ABpl_grn_dict,
    'ABplapp': ABpl_grn_dict,
    'P2': P2_grn_dict,
    'C': P2_grn_dict
}

grn_arrays = {}

for name, source_dict in grn_sources.items():
    grn = np.array(source_dict[name])
    grn_arrays[name] = grn  
    export_grn_edges(grn, name, regulatory_genes_name, target_genes_name, output_path)


In [ ]:
ABprp_inf_tar=pd.DataFrame(grn_arrays['ABprp']@np.array(ABprp_reg))
ABprap_inf_tar=pd.DataFrame(grn_arrays['ABprap']@np.array(ABprap_reg))
ABara_inf_tar=pd.DataFrame(grn_arrays['ABara']@np.array(ABara_reg))
ABaraap_inf_tar=pd.DataFrame(grn_arrays['ABaraap']@np.array(ABaraap_reg))
ABplpa_inf_tar=pd.DataFrame(grn_arrays['ABplpa']@np.array(ABplpa_reg))
ABalp_inf_tar=pd.DataFrame(grn_arrays['ABalp']@np.array(ABalp_reg))
ABprp_inf_tar.index=ABprap_inf_tar.index=ABara_inf_tar.index=ABaraap_inf_tar.index=ABplpa_inf_tar.index=ABalp_inf_tar.index=target_genes_name

In [ ]:
ABprp_high_expression_tar_infer=count_greater_than_threshold(ABprp_inf_tar,-count_matrices['ABprp_count_matrix'].values.mean())
ABprp_high_expression_tar=count_greater_than_threshold(ABprp_tar,-count_matrices['ABprp_count_matrix'].values.mean())

ABprap_high_expression_tar_infer=count_greater_than_threshold(ABprap_inf_tar,-count_matrices['ABprap_count_matrix'].values.mean())
ABprap_high_expression_tar=count_greater_than_threshold(ABprap_tar,-count_matrices['ABprap_count_matrix'].values.mean())

ABara_high_expression_tar_infer=count_greater_than_threshold(ABara_inf_tar,-count_matrices['ABara_count_matrix'].values.mean())
ABara_high_expression_tar=count_greater_than_threshold(ABara_tar,-count_matrices['ABara_count_matrix'].values.mean())

ABaraap_high_expression_tar_infer=count_greater_than_threshold(ABaraap_inf_tar,-count_matrices['ABaraap_count_matrix'].values.mean())
ABaraap_high_expression_tar=count_greater_than_threshold(ABaraap_tar,-count_matrices['ABaraap_count_matrix'].values.mean())


ABplpa_high_expression_tar_infer=count_greater_than_threshold(ABplpa_inf_tar,-count_matrices['ABplpa_count_matrix'].values.mean())
ABplpa_high_expression_tar=count_greater_than_threshold(ABplpa_tar,-count_matrices['ABplpa_count_matrix'].values.mean())


ABalp_high_expression_tar_infer = count_greater_than_threshold(ABalp_inf_tar,-count_matrices['ABalp_count_matrix'].values.mean())
ABalp_high_expression_tar = count_greater_than_threshold(ABalp_tar, -count_matrices['ABalp_count_matrix'].values.mean())

In [ ]:
data_inf = [ABalp_high_expression_tar_infer, ABplpa_high_expression_tar_infer, ABaraap_high_expression_tar_infer, ABara_high_expression_tar_infer, ABprap_high_expression_tar_infer, ABprp_high_expression_tar_infer]
data_real = [ABalp_high_expression_tar, ABplpa_high_expression_tar, ABaraap_high_expression_tar, ABara_high_expression_tar, ABprap_high_expression_tar, ABprp_high_expression_tar]

In [ ]:
norm_sums, loss_value, norm_value, perm_choice, sorted_perms = match_expression_data(data_inf, data_real, 1)
identity = tuple(range(len(data_inf)))
identity_idx = sorted_perms.index(identity) 

case_ids = np.arange(1, len(norm_sums) + 1)

plt.figure(figsize=(2.5, 1.5))

plt.bar(case_ids, norm_sums, color='#A8CDD9', width=1.0)

plt.bar(identity_idx + 1,
        norm_sums[identity_idx],
        color='red',
        edgecolor='darkred',
        linewidth=2.5,
        width=1.0)

plt.xlabel("Case ID (sorted by ascending loss)", fontsize=5, fontname='Arial')
plt.ylabel("Normalized loss", fontsize=5, fontname='Arial')
plt.xticks(ticks=np.arange(0, len(norm_sums)+1, 40), fontsize=5, fontname='Arial',rotation=90)
plt.yticks(ticks=np.linspace(0,1,6), fontsize=5, fontname='Arial')
plt.tick_params(axis='x', length=0.2)
plt.tick_params(axis='y', length=0.2)
plt.tight_layout()
plt.savefig(output_path+'matching_result_small.eps', format='eps', bbox_inches='tight')
plt.show()


In [ ]:
print("The minimum loss function:", loss_value)
print("The minimum loss function (normalized):", norm_value)
print("The permutation corresponding to the minimum loss function (column index selected for each row, counting starting from 0):", perm_choice)

#### Low confidence dataset verification

In [ ]:
ABplp_norm_count_matrix = normalize_to_zero_mean(count_matrices['ABplp_count_matrix'])
ABplp_tar = ABplp_norm_count_matrix.iloc[range(tar_num),:]
ABplp_reg = ABplp_norm_count_matrix.iloc[tar_num:,:]

ABplapp_norm_count_matrix = normalize_to_zero_mean(count_matrices['ABplapp_count_matrix'])
ABplapp_tar = ABplapp_norm_count_matrix.iloc[range(tar_num),:]
ABplapp_reg = ABplapp_norm_count_matrix.iloc[tar_num:,:]

ABprpa_norm_count_matrix = normalize_to_zero_mean(count_matrices['ABprpa_count_matrix'])
ABprpa_tar = ABprpa_norm_count_matrix.iloc[range(tar_num),:]
ABprpa_reg = ABprpa_norm_count_matrix.iloc[tar_num:,:]

ABala_norm_count_matrix = normalize_to_zero_mean(count_matrices['ABala_count_matrix'])
ABala_tar = ABala_norm_count_matrix.iloc[range(tar_num),:]
ABala_reg = ABala_norm_count_matrix.iloc[tar_num:,:]

ABarp_norm_count_matrix = normalize_to_zero_mean(count_matrices['ABarp_count_matrix'])
ABarp_tar = ABarp_norm_count_matrix.iloc[range(tar_num),:]
ABarp_reg = ABarp_norm_count_matrix.iloc[tar_num:,:]

ABarpa_norm_count_matrix = normalize_to_zero_mean(count_matrices['ABarpa_count_matrix'])
ABarpa_tar = ABarpa_norm_count_matrix.iloc[range(tar_num),:]
ABarpa_reg = ABarpa_norm_count_matrix.iloc[tar_num:,:]

ABal_norm_count_matrix = normalize_to_zero_mean(pd.concat([count_matrices['ABalp_count_matrix'], count_matrices['ABala_count_matrix']],axis=1))
ABal_tar = ABal_norm_count_matrix.iloc[range(tar_num),:]
ABal_reg = ABal_norm_count_matrix.iloc[tar_num:,:]

ABar_norm_count_matrix = normalize_to_zero_mean(pd.concat([count_matrices['ABarp_count_matrix'], count_matrices['ABara_count_matrix']],axis=1))
ABar_tar = ABar_norm_count_matrix.iloc[range(tar_num),:]
ABar_reg = ABar_norm_count_matrix.iloc[tar_num:,:]

ABaraa_norm_count_matrix = normalize_to_zero_mean(pd.concat([count_matrices['ABara_count_matrix'], count_matrices['ABaraap_count_matrix']],axis=1))
ABaraa_tar = ABaraa_norm_count_matrix.iloc[range(tar_num),:]
ABaraa_reg = ABaraa_norm_count_matrix.iloc[tar_num:,:]

In [ ]:
ABplp_inf_tar = pd.DataFrame(grn_arrays['ABplp']@np.array(ABplp_reg))
ABplapp_inf_tar = pd.DataFrame(grn_arrays['ABplapp']@np.array(ABplapp_reg))
ABprpa_inf_tar = pd.DataFrame(grn_arrays['ABprpa']@np.array(ABprpa_reg))
ABala_inf_tar = pd.DataFrame(grn_arrays['ABala']@np.array(ABala_reg))
ABarp_inf_tar = pd.DataFrame(grn_arrays['ABarp']@np.array(ABarp_reg))
ABarpa_inf_tar = pd.DataFrame(grn_arrays['ABarpa']@np.array(ABarpa_reg))
ABal_inf_tar = pd.DataFrame(grn_arrays['ABal']@np.array(ABal_reg))
ABar_inf_tar = pd.DataFrame(grn_arrays['ABar']@np.array(ABar_reg))
ABaraa_inf_tar = pd.DataFrame(grn_arrays['ABaraa']@np.array(ABaraa_reg))
ABplp_inf_tar.index = ABplapp_inf_tar.index = ABprpa_inf_tar.index = ABala_inf_tar.index = ABarp_inf_tar.index = ABarpa_inf_tar.index = ABal_inf_tar.index = ABar_inf_tar.index = ABaraa_inf_tar.index = target_genes_name

In [ ]:
ABplp_high_expression_tar_infer = count_greater_than_threshold(ABplp_inf_tar, -count_matrices['ABplp_count_matrix'].values.mean())
ABplp_high_expression_tar = count_greater_than_threshold(ABplp_tar, -count_matrices['ABplp_count_matrix'].values.mean())

ABplapp_high_expression_tar_infer = count_greater_than_threshold(ABplapp_inf_tar, -count_matrices['ABplapp_count_matrix'].values.mean())
ABplapp_high_expression_tar = count_greater_than_threshold(ABplapp_tar, -count_matrices['ABplapp_count_matrix'].values.mean())

ABprpa_high_expression_tar_infer = count_greater_than_threshold(ABprpa_inf_tar, -count_matrices['ABprpa_count_matrix'].values.mean())
ABprpa_high_expression_tar = count_greater_than_threshold(ABprpa_tar, -count_matrices['ABprpa_count_matrix'].values.mean())

ABala_high_expression_tar_infer = count_greater_than_threshold(ABala_inf_tar, -count_matrices['ABala_count_matrix'].values.mean())
ABala_high_expression_tar = count_greater_than_threshold(ABala_tar, -count_matrices['ABala_count_matrix'].values.mean())

ABarp_high_expression_tar_infer = count_greater_than_threshold(ABarp_inf_tar, -count_matrices['ABarp_count_matrix'].values.mean())
ABarp_high_expression_tar = count_greater_than_threshold(ABarp_tar, -count_matrices['ABarp_count_matrix'].values.mean())

ABarpa_high_expression_tar_infer = count_greater_than_threshold(ABarpa_inf_tar, -count_matrices['ABarpa_count_matrix'].values.mean())
ABarpa_high_expression_tar = count_greater_than_threshold(ABarpa_tar, -count_matrices['ABarpa_count_matrix'].values.mean())

ABal_high_expression_tar_infer = count_greater_than_threshold(ABal_inf_tar, -pd.concat([count_matrices['ABalp_count_matrix'], count_matrices['ABala_count_matrix']],axis=1).values.mean())
ABal_high_expression_tar = count_greater_than_threshold(ABal_tar, -pd.concat([count_matrices['ABalp_count_matrix'], count_matrices['ABala_count_matrix']],axis=1).values.mean())

ABar_high_expression_tar_infer = count_greater_than_threshold(ABar_inf_tar, -pd.concat([count_matrices['ABarp_count_matrix'], count_matrices['ABara_count_matrix']],axis=1).values.mean())
ABar_high_expression_tar = count_greater_than_threshold(ABar_tar, -pd.concat([count_matrices['ABarp_count_matrix'], count_matrices['ABara_count_matrix']],axis=1).values.mean())

ABaraa_high_expression_tar_infer = count_greater_than_threshold(ABaraa_inf_tar, -pd.concat([count_matrices['ABara_count_matrix'], count_matrices['ABaraap_count_matrix']],axis=1).values.mean())
ABaraa_high_expression_tar = count_greater_than_threshold(ABaraa_tar, -pd.concat([count_matrices['ABara_count_matrix'], count_matrices['ABaraap_count_matrix']],axis=1).values.mean())

In [ ]:
data_inf = [ABalp_high_expression_tar_infer, ABplpa_high_expression_tar_infer, ABaraap_high_expression_tar_infer, ABara_high_expression_tar_infer, ABprap_high_expression_tar_infer, ABprp_high_expression_tar_infer,
           ABplp_high_expression_tar_infer, ABplapp_high_expression_tar_infer, ABprpa_high_expression_tar_infer, ABala_high_expression_tar_infer, ABarp_high_expression_tar_infer, ABarpa_high_expression_tar_infer,
           ABal_high_expression_tar_infer, ABar_high_expression_tar_infer, ABaraa_high_expression_tar_infer]
data_real = [ABalp_high_expression_tar, ABplpa_high_expression_tar, ABaraap_high_expression_tar, ABara_high_expression_tar, ABprap_high_expression_tar, ABprp_high_expression_tar,
            ABplp_high_expression_tar, ABplapp_high_expression_tar, ABprpa_high_expression_tar, ABala_high_expression_tar, ABarp_high_expression_tar, ABarpa_high_expression_tar,
            ABal_high_expression_tar, ABar_high_expression_tar, ABaraa_high_expression_tar]

In [ ]:
norm_sums, loss_value, norm_value, perm_choice, sorted_perms = match_expression_data(data_inf, data_real, 1, 10000)
p_value = [i/10000 for i in range(0,10000) if sorted_perms[i]==tuple(range(0,15))]
print("The p value is:", p_value[0])

identity = tuple(range(len(data_inf)))
identity_idx = sorted_perms.index(identity) 


case_ids = np.arange(1, len(norm_sums) + 1)

plt.figure(figsize=(2.5, 1.5))

plt.bar(case_ids, norm_sums, color='#A8CDD9', width=1.0)

plt.bar(identity_idx + 1,
        norm_sums[identity_idx],
        color='red',
        edgecolor='darkred',
        linewidth=2.5,
        width=1.0)

plt.xlabel("Case ID (sorted by ascending loss)", fontsize=5, fontname='Arial')
plt.ylabel("Normalized loss", fontsize=5, fontname='Arial')
plt.xticks(ticks=np.arange(0, len(norm_sums)+1, 500), fontsize=5, fontname='Arial',rotation=90)
plt.yticks(ticks=np.linspace(0,1,6), fontsize=5, fontname='Arial')
plt.tick_params(axis='x', length=0.2)
plt.tick_params(axis='y', length=0.2)
plt.tight_layout()
plt.savefig(output_path+'matching_result_big.eps', format='eps', bbox_inches='tight')
plt.show()


## Data analysis on C.elegans dataset

### Revealing the reconfiguration pattern of inferred GRNs along cell lineages

In [ ]:
atac_file_path = "examples/data/C.elegans/C.elegans_Full/input_data/EMS/atac_data.csv"
expression_file_path = "examples/data/C.elegans/C.elegans_Full/input_data/EMS/expression_data.csv"
fate_map_path ='examples/data/C.elegans/C.elegans_Full/input_data/EMS/edge_dict_EMS.csv'

saved_dir = 'examples/results/C.elegans/C.elegans_Full/EMS'

edge_dict = load_fate_map_topology(fate_map_path)
edge = parse_edge_dict(edge_dict)
fate_map = FateMap(edge)

regulator_names = list(pd.read_csv('examples/data/C.elegans/C.elegans_Full/additional_input_data/regulatory_genes_name.csv',index_col=0).iloc[:,0])
target_gene_names = list(pd.read_csv('examples/data/C.elegans/C.elegans_Full/additional_input_data/target_genes_name.csv',index_col=0).iloc[:,0])

dynamic_networks_dict = get_dynamic_networks(saved_dir, fate_map, 0.1, regulator_names, target_gene_names)

In [ ]:
output_path = 'examples/results/C.elegans/C.elegans_Full/figures/'
path = list(fate_map.nodes.keys())

In [ ]:
plot_regulatory_interactions_along_fate_map(dynamic_networks_dict, path, output_path)

In [ ]:
plot_dynamic_regulator_count(saved_dir, regulator_names,'elt-7', path, 0.05, output_path)

In [ ]:
plot_regulatory_genes_along_fate_map(saved_dir, regulator_names, 'elt-7', path, 0.05, output_path)

In [ ]:
plot_target_genes_along_fate_map(target_gene_names, regulator_names, 'negative', saved_dir, path, 0.1, output_path)

In [ ]:
plot_target_genes_along_fate_map(target_gene_names, regulator_names,'positive', saved_dir, path, 0.1, output_path)

In [ ]:
plot_regulatory_network_along_fate_map('elt-7', dynamic_networks_dict, path, output_path, 0.1, figsize=(19.5,2.5))

### Identifying regulatory genes key for cell differentiation

In [ ]:
groups = {
    'ABar': {
        'names': ['ABarpp-', 'ABarap-', 'ABarpapp-', 'ABarpapa-', 'ABaraaa-', 'ABaraapp-', 'ABaraapaa-'],
        'extra_keys': ['ABarp','ABarpa','ABara','ABaraa','ABaraap','ABar']
    },
    'ABpr': {
        'names': ['ABprppa-','ABprpap-','ABprpaa-','ABpraa-','ABprappa-','ABprapa-'],
        'extra_keys': ['ABprp','ABprpa','ABprapp','ABpra','ABpr','ABprap']
    },
    'ABpl': {
        'names': ['ABplpppp-','ABplpap-','ABplpaa-','ABplapap-','ABplappp-','ABplappa-'],
        'extra_keys': ['ABplp','ABplpa','ABpla','ABplapp','ABpl']
    },
    'ABal': {
        'names': ['ABalpp-','ABalpa-','ABalap-','ABalaa-'],
        'extra_keys': ['ABalp','ABala','ABal']
    },
    'EMS': {
        'names': ['MSa-','MSp-','Ea-','Ep-'],
        'extra_keys': ['MS','E','EMS']
    },
    'P2': {
        'names': ['D-','hyp','body_muscle'],
        'extra_keys': ['P2','C']
    }
}

base_path = 'examples/data/C.elegans/C.elegans_Full/additional_input_data'
gene_expression_matrices = {}
high_expression_genes = {}

for group, info in groups.items():
    group_dict = {}
    for name in info['names']:
        file_path = f'{base_path}/{name}_count_matrix.csv'
        df = pd.read_csv(file_path, index_col=0)
        gene_expression_matrices[name] = df
        genes_var_name = name
        group_dict[genes_var_name] = [i for i in get_high_expression_genes(df) if i in target_genes_name]

    for key in info['extra_keys']:
        group_dict[key] = None
        
    high_expression_genes[group] = group_dict


In [ ]:
key_regulators_df_concat = pd.DataFrame()
for lineage in edges_dict_Celegans.keys():
    edges = parse_edge_dict(edges_dict_Celegans[lineage])
    fate_map = FateMap(edges,high_expression_genes[lineage])
    nodes = fate_map.node_internals[::-1]+[fate_map.node_root]
    for node in nodes:
        input_path = os.path.join('examples/results/C.elegans/C.elegans_Full', lineage)
        key_regulators = identify_differentiation_regulators(input_path, fate_map, 0, regulatory_genes_name, target_genes_name, node)
        high_expression_genes_of_ancestor_node = list(set(key_regulators.loc[((key_regulators['DRS']>0.05) & (key_regulators['DRC']>0.5)), 'regulator_id']) & set(target_genes_name))
        high_expression_genes[lineage][node] = high_expression_genes_of_ancestor_node
        fate_map = FateMap(edges,high_expression_genes[lineage])
        key_regulators_df_concat = pd.concat([key_regulators_df_concat,key_regulators])

In [ ]:
nodes = list(key_regulators_df_concat['node_id'].drop_duplicates())
plot_key_genes_differentiation(key_regulators_df_concat.fillna(0), nodes, regulatory_genes_name, output_path, figsize=(20,5))

### Identifying regulatory genes key for cell fate decision

##### P2

In [ ]:
edges = parse_edge_dict(edges_dict_Celegans['P2'])
fate_map= FateMap(edges, high_expression_genes['P2'])
input_path = 'examples/results/C.elegans/C.elegans_Full/P2'
grn_dict_P2 = get_dynamic_networks(input_path, fate_map, 0.1, regulatory_genes_name, target_genes_name)

In [ ]:
ancestor_node_id = 'C'
child_nodes = [fate_map.nodes[ancestor_node_id].directed_edges[i].end for i in range(2)]
print(child_nodes)
high_expression_target_genes_in_child_1 = fate_map.nodes[child_nodes[0]].high_expression_genes_in_leaf
high_expression_target_genes_in_child_2 = fate_map.nodes[child_nodes[1]].high_expression_genes_in_leaf
 
target_genes_1 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_1) - set(high_expression_target_genes_in_child_2))
target_genes_2 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_2) - set(high_expression_target_genes_in_child_1))

key_regulators_df_C = identify_fate_bias_regulators(grn_dict_P2, 'C', target_genes_1, target_genes_2, regulatory_genes_name)

df = merge_regulator_scores(key_regulators_df_C, child_nodes)
df_filter = filter_regulators(df, regulatory_genes_name)

plot_key_genes_fate_bias(df_filter, child_nodes, output_path, figsize=(1.65,10))

##### EMS

In [ ]:
edges = parse_edge_dict(edges_dict_Celegans['EMS'])
fate_map = FateMap(edges,high_expression_genes['EMS'])
input_path = 'examples/results/C.elegans/C.elegans_Full/EMS'
grn_dict_EMS = get_dynamic_networks(input_path, fate_map, 0.1, regulatory_genes_name, target_genes_name)

In [ ]:
ancestor_node_id = 'MS'
child_nodes = [fate_map.nodes[ancestor_node_id].directed_edges[i].end for i in range(2)]
print(child_nodes)
high_expression_target_genes_in_child_1 = fate_map.nodes[child_nodes[0]].high_expression_genes_in_leaf
high_expression_target_genes_in_child_2 = fate_map.nodes[child_nodes[1]].high_expression_genes_in_leaf
 
target_genes_1 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_1) - set(high_expression_target_genes_in_child_2))
target_genes_2 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_2) - set(high_expression_target_genes_in_child_1))

key_regulators_df_MS = identify_fate_bias_regulators(grn_dict_EMS, 'MS', target_genes_1, target_genes_2, regulatory_genes_name)

df = merge_regulator_scores(key_regulators_df_MS, child_nodes)
df_filter = filter_regulators(df, regulatory_genes_name)
plot_key_genes_fate_bias(df_filter, child_nodes, output_path, figsize=(1.65,10))

In [ ]:
ancestor_node_id = 'E'
child_nodes = [fate_map.nodes[ancestor_node_id].directed_edges[i].end for i in range(2)]
print(child_nodes)
high_expression_target_genes_in_child_1 = fate_map.nodes[child_nodes[0]].high_expression_genes_in_leaf
high_expression_target_genes_in_child_2 = fate_map.nodes[child_nodes[1]].high_expression_genes_in_leaf
 
target_genes_1 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_1) - set(high_expression_target_genes_in_child_2))
target_genes_2 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_2) - set(high_expression_target_genes_in_child_1))

key_regulators_df_E = identify_fate_bias_regulators(grn_dict_EMS, 'E', target_genes_1, target_genes_2, regulatory_genes_name)

df = merge_regulator_scores(key_regulators_df_E, child_nodes)
df_filter = filter_regulators(df, regulatory_genes_name)

plot_key_genes_fate_bias(df_filter, child_nodes, output_path, figsize=(1.65,10))

##### ABpr

In [ ]:
edges = parse_edge_dict(edges_dict_Celegans['ABpr'])
fate_map = FateMap(edges,high_expression_genes['ABpr'])
input_path ='examples/results/C.elegans/C.elegans_Full/ABpr'
grn_dict_ABpr = get_dynamic_networks(input_path, fate_map, 0.1, regulatory_genes_name, target_genes_name)

In [ ]:
ancestor_node_id = 'ABprpa'
child_nodes = [fate_map.nodes[ancestor_node_id].directed_edges[i].end for i in range(2)]
print(child_nodes)
high_expression_target_genes_in_child_1 = fate_map.nodes[child_nodes[0]].high_expression_genes_in_leaf
high_expression_target_genes_in_child_2 = fate_map.nodes[child_nodes[1]].high_expression_genes_in_leaf
 
target_genes_1 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_1) - set(high_expression_target_genes_in_child_2))
target_genes_2 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_2) - set(high_expression_target_genes_in_child_1))

key_regulators_df_ABprpa = identify_fate_bias_regulators(grn_dict_ABpr, 'ABprpa', target_genes_1, target_genes_2, regulatory_genes_name)

df = merge_regulator_scores(key_regulators_df_ABprpa, child_nodes)
df_filter = filter_regulators(df, regulatory_genes_name)

plot_key_genes_fate_bias(df_filter, child_nodes, output_path, figsize=(1.65,10))

In [ ]:
ancestor_node_id = 'ABprap'
child_nodes = [fate_map.nodes[ancestor_node_id].directed_edges[i].end for i in range(2)]
print(child_nodes)
high_expression_target_genes_in_child_1 = fate_map.nodes[child_nodes[0]].high_expression_genes_in_leaf
high_expression_target_genes_in_child_2 = fate_map.nodes[child_nodes[1]].high_expression_genes_in_leaf
 
target_genes_1 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_1) - set(high_expression_target_genes_in_child_2))
target_genes_2 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_2) - set(high_expression_target_genes_in_child_1))

key_regulators_df_ABprpa = identify_fate_bias_regulators(grn_dict_ABpr, 'ABprap', target_genes_1, target_genes_2, regulatory_genes_name)

df = merge_regulator_scores(key_regulators_df_ABprpa, child_nodes)
df_filter = filter_regulators(df, regulatory_genes_name)

plot_key_genes_fate_bias(df_filter, child_nodes, output_path, figsize=(1.65,10))

##### ABpl

In [ ]:
edges = parse_edge_dict(edges_dict_Celegans['ABpl'])
fate_map = FateMap(edges,high_expression_genes['ABpl'])
input_path = 'examples/results/C.elegans/C.elegans_Full/ABpl'
grn_dict_ABpl = get_dynamic_networks(input_path, fate_map, 0.1, regulatory_genes_name, target_genes_name)

In [ ]:
ancestor_node_id = 'ABplpa'
child_nodes = [fate_map.nodes[ancestor_node_id].directed_edges[i].end for i in range(2)]
print(child_nodes)
high_expression_target_genes_in_child_1 = fate_map.nodes[child_nodes[0]].high_expression_genes_in_leaf
high_expression_target_genes_in_child_2 = fate_map.nodes[child_nodes[1]].high_expression_genes_in_leaf
 
target_genes_1 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_1) - set(high_expression_target_genes_in_child_2))
target_genes_2 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_2) - set(high_expression_target_genes_in_child_1))

key_regulators_df_ABplpa = identify_fate_bias_regulators(grn_dict_ABpl, 'ABplpa', target_genes_1, target_genes_2, regulatory_genes_name)

df = merge_regulator_scores(key_regulators_df_ABplpa, child_nodes)
df_filter = filter_regulators(df, regulatory_genes_name)
plot_key_genes_fate_bias(df_filter, child_nodes, output_path, figsize=(1.65,10))

In [ ]:
ancestor_node_id = 'ABplapp'
child_nodes = [fate_map.nodes[ancestor_node_id].directed_edges[i].end for i in range(2)]
print(child_nodes)
high_expression_target_genes_in_child_1 = fate_map.nodes[child_nodes[0]].high_expression_genes_in_leaf
high_expression_target_genes_in_child_2 = fate_map.nodes[child_nodes[1]].high_expression_genes_in_leaf
 
target_genes_1 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_1) - set(high_expression_target_genes_in_child_2))
target_genes_2 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_2) - set(high_expression_target_genes_in_child_1))


key_regulators_df_ABplapp = identify_fate_bias_regulators(grn_dict_ABpl, 'ABplapp', target_genes_1, target_genes_2, regulatory_genes_name)

df = merge_regulator_scores(key_regulators_df_ABplapp, child_nodes)
df_filter = filter_regulators(df,regulatory_genes_name)
plot_key_genes_fate_bias(df_filter, child_nodes, output_path, figsize=(1.65,10))

##### ABal

In [ ]:
edges = parse_edge_dict(edges_dict_Celegans['ABal'])
fate_map = FateMap(edges, high_expression_genes['ABal'])
input_path = 'examples/results/C.elegans/C.elegans_Full/ABal'
grn_dict_ABal = get_dynamic_networks(input_path, fate_map, 0.1, regulatory_genes_name, target_genes_name)

In [ ]:
ancestor_node_id = 'ABalp'
child_nodes = [fate_map.nodes[ancestor_node_id].directed_edges[i].end for i in range(2)]
print(child_nodes)
high_expression_target_genes_in_child_1 = fate_map.nodes[child_nodes[0]].high_expression_genes_in_leaf
high_expression_target_genes_in_child_2 = fate_map.nodes[child_nodes[1]].high_expression_genes_in_leaf
 
target_genes_1 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_1) - set(high_expression_target_genes_in_child_2))
target_genes_2 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_2) - set(high_expression_target_genes_in_child_1))

key_regulators_df_ABalp = identify_fate_bias_regulators(grn_dict_ABal, 'ABalp', target_genes_1, target_genes_2, regulatory_genes_name)

df = merge_regulator_scores(key_regulators_df_ABalp, child_nodes)
df_filter = filter_regulators(df, regulatory_genes_name)
plot_key_genes_fate_bias(df_filter, child_nodes, output_path, figsize=(1.65,10))

In [ ]:
ancestor_node_id = 'ABala'
child_nodes = [fate_map.nodes[ancestor_node_id].directed_edges[i].end for i in range(2)]
print(child_nodes)
high_expression_target_genes_in_child_1 = fate_map.nodes[child_nodes[0]].high_expression_genes_in_leaf
high_expression_target_genes_in_child_2 = fate_map.nodes[child_nodes[1]].high_expression_genes_in_leaf

target_genes_1 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_1) - set(high_expression_target_genes_in_child_2))
target_genes_2 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_2) - set(high_expression_target_genes_in_child_1))

key_regulators_df_ABalp = identify_fate_bias_regulators(grn_dict_ABal, 'ABala', target_genes_1, target_genes_2, regulatory_genes_name)

df = merge_regulator_scores(key_regulators_df_ABalp, child_nodes)
df_filter = filter_regulators(df, regulatory_genes_name)
plot_key_genes_fate_bias(df_filter, child_nodes, output_path, figsize=(1.65,10))

##### ABar

In [ ]:
edges = parse_edge_dict(edges_dict_Celegans['ABar'])
fate_map = FateMap(edges,high_expression_genes['ABar'])
input_path = 'examples/results/C.elegans/C.elegans_Full/ABar'
grn_dict_ABar = get_dynamic_networks(input_path,fate_map, 0.1, regulatory_genes_name, target_genes_name)

In [ ]:
ancestor_node_id = 'ABarpa'
child_nodes = [fate_map.nodes[ancestor_node_id].directed_edges[i].end for i in range(2)]
print(child_nodes)
high_expression_target_genes_in_child_1 = fate_map.nodes[child_nodes[0]].high_expression_genes_in_leaf
high_expression_target_genes_in_child_2 = fate_map.nodes[child_nodes[1]].high_expression_genes_in_leaf
 
target_genes_1 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_1) - set(high_expression_target_genes_in_child_2))
target_genes_2 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_2) - set(high_expression_target_genes_in_child_1))

key_regulators_df_ABalp = identify_fate_bias_regulators(grn_dict_ABar, 'ABarpa', target_genes_1, target_genes_2, regulatory_genes_name)

df = merge_regulator_scores(key_regulators_df_ABalp, child_nodes)
df_filter = filter_regulators(df, regulatory_genes_name)
plot_key_genes_fate_bias(df_filter, child_nodes, output_path, figsize=(1.65,10))

In [ ]:
ancestor_node_id = 'ABaraap'
child_nodes = [fate_map.nodes[ancestor_node_id].directed_edges[i].end for i in range(2)]
print(child_nodes)
high_expression_target_genes_in_child_1 = fate_map.nodes[child_nodes[0]].high_expression_genes_in_leaf
high_expression_target_genes_in_child_2 = fate_map.nodes[child_nodes[1]].high_expression_genes_in_leaf
 
target_genes_1 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_1) - set(high_expression_target_genes_in_child_2))
target_genes_2 = list(set(target_genes_name) & set(high_expression_target_genes_in_child_2) - set(high_expression_target_genes_in_child_1))

key_regulators_df_ABalp = identify_fate_bias_regulators(grn_dict_ABar,'ABaraap', target_genes_1, target_genes_2, regulatory_genes_name)

df = merge_regulator_scores(key_regulators_df_ABalp, child_nodes)
df_filter = filter_regulators(df, regulatory_genes_name)
plot_key_genes_fate_bias(df_filter, child_nodes, output_path, figsize=(1.65,10))

### Identifying constitutive and specific regulatory interactions

In [ ]:
saved_dir_EMS = 'examples/results/C.elegans/C.elegans_Full/EMS'
edges_EMS = parse_edge_dict(edges_dict_Celegans['EMS'])
fate_map_EMS = FateMap(edges_EMS)
X, centers, weight_matrix_raw=cluster_regulatory_interactions(saved_dir_EMS, fate_map_EMS, 0.3, regulatory_genes_name, target_genes_name, len(regulatory_genes_name),len(target_genes_name), 5, 2)

In [ ]:
edges_cluster_to_nodes = summarize_interaction_cluster_specificity(saved_dir_EMS, fate_map_EMS, 0.1, regulatory_genes_name, target_genes_name, 5, 0.3, X, len(regulatory_genes_name), len(target_genes_name),weight_matrix_raw)

In [ ]:
plot_regulatory_interactions_by_cell_type(edges_cluster_to_nodes, output_path)